 data_loader.py + feature_pipeline.py + trainer.py; FastAPI /predict; ONNX assert; MLflow 3 runs; batch_predict.py; end_to_end_check()
function

In [2]:

import pickle
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load dataset
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Create and train model
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000))
])

model.fit(X_train, y_train)

# Save trained model
with open("/content/model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save input features without target labels
X.to_csv("/content/batch_input.csv", index=False)

print("Model saved: /content/model.pkl")
print("Dataset saved: /content/batch_input.csv")
print("Training completed successfully!")


Model saved: /content/model.pkl
Dataset saved: /content/batch_input.csv
Training completed successfully!


Write end_to_end_check(model_path, data_path, n_batch=20) — loads model, samples n_batch rows, asserts predictions ∈ {0,1} and
probabilities ∈ [0,1], prints 'Pipeline check PASSED'

In [4]:

import pickle
import numpy as np
import pandas as pd

def end_to_end_check(model_path, data_path, n_batch=20):

    # Load model
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    # Load dataset
    data = pd.read_csv(data_path)

    assert n_batch > 0, "n_batch must be positive"
    assert len(data) >= n_batch, "Not enough rows in dataset"

    # Sample rows
    batch = data.sample(n=n_batch, random_state=42)

    # Match expected feature columns
    if hasattr(model, "feature_names_in_"):
        batch = batch.loc[:, model.feature_names_in_]

    # Generate predictions and probabilities
    predictions = model.predict(batch)
    probabilities = model.predict_proba(batch)

    # Validate predictions
    assert set(np.unique(predictions)).issubset({0, 1}), \
        "Predictions must be 0 or 1"

    # Validate probabilities
    assert np.isfinite(probabilities).all(), \
        "Probabilities contain invalid values"

    assert np.all((probabilities >= 0) & (probabilities <= 1)), \
        "Probabilities must be between 0 and 1"

    assert np.allclose(probabilities.sum(axis=1), 1.0), \
        "Probabilities must sum to 1"

    print("Pipeline check PASSED")

    return predictions, probabilities


# Execute the check
predictions, probabilities = end_to_end_check(
    model_path="/content/model.pkl",
    data_path="/content/batch_input.csv",
    n_batch=20
)

print("Predictions:", predictions)
print("Probability shape:", probabilities.shape)


Pipeline check PASSED
Predictions: [1 0 0 1 1 0 0 0 1 1 1 0 1 0 1 0 1 1 1 0]
Probability shape: (20, 2)
